<a href="https://colab.research.google.com/github/dystaSatria/Deep-Learning/blob/main/Internship%20Projects/F1Skoru/fiskoru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# F1 SKORU HESAPLAMA ÖRNEKLERİ
# =====================================

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. MANuel F1 Skoru Hesaplama Fonksiyonu
def hesapla_f1_skoru(y_true, y_pred):
    """
    F1 skorunu manuel olarak hesaplayan fonksiyon

    Args:
        y_true: Gerçek etiketler
        y_pred: Tahmin edilen etiketler

    Returns:
        f1_skoru, hassasiyet, hatırlama değerleri
    """
    # True Positive, False Positive, False Negative hesaplama
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    # Hassasiyet (Precision) hesaplama
    hassasiyet = tp / (tp + fp) if (tp + fp) > 0 else 0

    # Hatırlama (Recall) hesaplama
    hatirlama = tp / (tp + fn) if (tp + fn) > 0 else 0

    # F1 Skoru hesaplama
    f1 = 2 * (hassasiyet * hatirlama) / (hassasiyet + hatirlama) if (hassasiyet + hatirlama) > 0 else 0

    return f1, hassasiyet, hatirlama

# 2. Örnek Veri ile F1 Skoru Demonstration
def f1_ornek_demonstrasyonu():
    """Tıbbi teşhis örneği ile F1 skoru demonstrasyonu"""

    # Simüle edilmiş tıbbi teşhis verileri
    # 1: Kanser var, 0: Kanser yok
    gercek_etiketler = np.array([1, 1, 0, 1, 0, 0, 1, 1, 0, 0,
                                1, 0, 1, 0, 1, 1, 0, 0, 1, 0])

    # Model tahminleri (3 farklı model performansı)
    model_1_tahmin = np.array([1, 1, 0, 1, 0, 1, 1, 1, 0, 0,
                              1, 0, 1, 0, 1, 0, 0, 0, 1, 0])

    model_2_tahmin = np.array([1, 0, 0, 1, 0, 0, 1, 1, 1, 0,
                              1, 0, 0, 0, 1, 1, 0, 0, 1, 0])

    model_3_tahmin = np.array([1, 1, 0, 1, 0, 0, 1, 1, 0, 0,
                              1, 0, 1, 0, 1, 1, 0, 0, 1, 0])

    print("=== TIBBİ TEŞHİS SİSTEMİ F1 SKORU ANALİZİ ===\n")

    modeller = [model_1_tahmin, model_2_tahmin, model_3_tahmin]
    model_isimleri = ["Agresif Model", "Konservatif Model", "Dengeli Model"]

    for i, (model_tahmin, model_ismi) in enumerate(zip(modeller, model_isimleri)):
        f1, hassasiyet, hatirlama = hesapla_f1_skoru(gercek_etiketler, model_tahmin)

        print(f"{model_ismi}:")
        print(f"  Hassasiyet (Precision): {hassasiyet:.3f}")
        print(f"  Hatırlama (Recall): {hatirlama:.3f}")
        print(f"  F1 Skoru: {f1:.3f}")

        # Sklearn ile doğrulama
        sklearn_f1 = f1_score(gercek_etiketler, model_tahmin)
        print(f"  Sklearn F1 (Doğrulama): {sklearn_f1:.3f}")
        print("-" * 40)

# 3. DENSENET IMPLEMENTASYonu
# ===========================

class DenseBlock(layers.Layer):
    """DenseNet'in temel yapı taşı olan Dense Block"""

    def __init__(self, num_layers, growth_rate, **kwargs):
        super(DenseBlock, self).__init__(**kwargs)
        self.num_layers = num_layers
        self.growth_rate = growth_rate
        self.layers_list = []

        # Her katman için composite function oluştur
        for i in range(num_layers):
            layer_block = [
                layers.BatchNormalization(),
                layers.ReLU(),
                layers.Conv2D(4 * growth_rate, 1, use_bias=False),  # Bottleneck
                layers.BatchNormalization(),
                layers.ReLU(),
                layers.Conv2D(growth_rate, 3, padding='same', use_bias=False)
            ]
            self.layers_list.append(layer_block)

    def call(self, x):
        features = [x]

        for layer_block in self.layers_list:
            # Önceki tüm katmanların çıktılarını birleştir
            concat_features = layers.Concatenate()(features)

            # Composite function uygula
            out = concat_features
            for layer in layer_block:
                out = layer(out)

            features.append(out)

        # Son durumda tüm feature'ları birleştir
        return layers.Concatenate()(features)

class TransitionLayer(layers.Layer):
    """Dense block'lar arasındaki geçiş katmanı"""

    def __init__(self, compression_factor=0.5, **kwargs):
        super(TransitionLayer, self).__init__(**kwargs)
        self.compression_factor = compression_factor

    def build(self, input_shape):
        num_filters = int(input_shape[-1] * self.compression_factor)

        self.bn = layers.BatchNormalization()
        self.relu = layers.ReLU()
        self.conv = layers.Conv2D(num_filters, 1, use_bias=False)
        self.pool = layers.AveragePooling2D(2, strides=2)

    def call(self, x):
        x = self.bn(x)
        x = self.relu(x)
        x = self.conv(x)
        x = self.pool(x)
        return x

def create_densenet121(input_shape=(224, 224, 3), num_classes=1000):
    """
    DenseNet-121 modelini oluşturan fonksiyon

    Args:
        input_shape: Giriş görüntüsünün boyutu
        num_classes: Sınıf sayısı

    Returns:
        DenseNet-121 modeli
    """

    inputs = layers.Input(shape=input_shape)

    # İlk konvolüsyon katmanı
    x = layers.Conv2D(64, 7, strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(3, strides=2, padding='same')(x)

    # Dense Block konfigürasyonu (DenseNet-121 için)
    dense_blocks_config = [6, 12, 24, 16]  # Her block'taki katman sayıları
    growth_rate = 32

    # Dense Block'ları ve Transition Layer'ları ekle
    for i, num_layers in enumerate(dense_blocks_config):
        # Dense Block ekle
        dense_block = DenseBlock(num_layers, growth_rate)
        x = dense_block(x)

        # Son block hariç Transition Layer ekle
        if i < len(dense_blocks_config) - 1:
            transition = TransitionLayer()
            x = transition(x)

    # Sınıflandırma katmanları
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling2D()(x)

    # Çıkış katmanı
    if num_classes == 1:
        # Binary sınıflandırma için
        outputs = layers.Dense(1, activation='sigmoid')(x)
    else:
        # Multi-class sınıflandırma için
        outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='DenseNet121')
    return model

# 4. DenseNet Eğitim Örneği
def densenet_egitim_ornegi():
    """DenseNet ile basit bir sınıflandırma örneği"""

    print("=== DENSENET EĞİTİM ÖRNEĞİ ===\n")

    # Küçük boyutlu model oluştur (demo için)
    model = create_densenet121(input_shape=(64, 64, 3), num_classes=10)

    # Model özetini göster
    print("Model Mimarisi:")
    model.summary()

    # Model derleme
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # Simüle edilmiş veri oluştur
    X_train = np.random.randn(100, 64, 64, 3)
    y_train = np.random.randint(0, 10, (100,))
    X_val = np.random.randn(20, 64, 64, 3)
    y_val = np.random.randint(0, 10, (20,))

    print(f"\nEğitim verisi boyutu: {X_train.shape}")
    print(f"Doğrulama verisi boyutu: {X_val.shape}")

    # Model eğitimi (kısa demo)
    print("\nModel eğitimi başlıyor...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=2,  # Demo için kısa
        batch_size=16,
        verbose=1
    )

    # Tahmin yap ve F1 skoru hesapla
    y_pred = model.predict(X_val)
    y_pred_classes = np.argmax(y_pred, axis=1)

    # Multi-class için F1 skoru
    f1_macro = f1_score(y_val, y_pred_classes, average='macro')
    f1_weighted = f1_score(y_val, y_pred_classes, average='weighted')

    print(f"\nDoğrulama F1 Skoru (Macro): {f1_macro:.3f}")
    print(f"Doğrulama F1 Skoru (Weighted): {f1_weighted:.3f}")

# 5. Karşılaştırmalı Model Analizi
def model_karsilastirma():
    """DenseNet vs ResNet karşılaştırması için fonksiyon şablonu"""

    print("=== MODEL KARŞILAŞTIRMA ANALİZİ ===\n")

    # DenseNet-121
    densenet = create_densenet121(input_shape=(224, 224, 3), num_classes=1000)

    # ResNet-50 (karşılaştırma için)
    resnet = keras.applications.ResNet50(
        input_shape=(224, 224, 3),
        include_top=True,
        weights=None,
        classes=1000
    )

    print("Model Parametre Karşılaştırması:")
    print(f"DenseNet-121 parametreleri: {densenet.count_params():,}")
    print(f"ResNet-50 parametreleri: {resnet.count_params():,}")

    parameter_farki = ((resnet.count_params() - densenet.count_params()) / resnet.count_params()) * 100
    print(f"DenseNet, ResNet-50'den %{parameter_farki:.1f} daha az parametre kullanıyor")

# Ana fonksiyon - tüm örnekleri çalıştır
if __name__ == "__main__":
    print("F1 SKORU VE DENSENET KAPSAMLI ÖRNEKLERİ")
    print("=" * 50)

    # F1 skoru demonstrasyonu
    f1_ornek_demonstrasyonu()

    print("\n" + "=" * 50 + "\n")

    # DenseNet eğitim örneği
    densenet_egitim_ornegi()

    print("\n" + "=" * 50 + "\n")

    # Model karşılaştırması
    model_karsilastirma()

    print("\nTüm örnekler tamamlandı! 🎉")

F1 SKORU VE DENSENET KAPSAMLI ÖRNEKLERİ
=== TIBBİ TEŞHİS SİSTEMİ F1 SKORU ANALİZİ ===

Agresif Model:
  Hassasiyet (Precision): 0.900
  Hatırlama (Recall): 0.900
  F1 Skoru: 0.900
  Sklearn F1 (Doğrulama): 0.900
----------------------------------------
Konservatif Model:
  Hassasiyet (Precision): 0.889
  Hatırlama (Recall): 0.800
  F1 Skoru: 0.842
  Sklearn F1 (Doğrulama): 0.842
----------------------------------------
Dengeli Model:
  Hassasiyet (Precision): 1.000
  Hatırlama (Recall): 1.000
  F1 Skoru: 1.000
  Sklearn F1 (Doğrulama): 1.000
----------------------------------------


=== DENSENET EĞİTİM ÖRNEĞİ ===

Model Mimarisi:


Model: "DenseNet121"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 32, 32, 64)     │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_block (DenseBlock)        │ (None, 16, 16, 256)    │       338,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transition_layer                │ (None, 8, 8, 128)      │        33,792 │
│ (TransitionLayer)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_block_1 (DenseBlock)      │ (None, 8, 8, 512)      │       930,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transition_layer_1              │ (None, 4, 4, 256)      │       133,120 │
│ (TransitionLayer)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_block_2 (DenseBlock)      │ (None, 4, 4, 1024)     │     2,873,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transition_layer_2              │ (None, 2, 2, 512)      │       528,384 │
│ (TransitionLayer)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_block_3 (DenseBlock)      │ (None, 2, 2, 1024)     │     2,186,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_120         │ (None, 2, 2, 1024)     │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_120 (ReLU)                │ (None, 2, 2, 1024)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        10,250 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,047,754 (26.89 MB)

 Trainable params: 6,964,106 (26.57 MB)

 Non-trainable params: 83,648 (326.75 KB)


Eğitim verisi boyutu: (100, 64, 64, 3)
Doğrulama verisi boyutu: (20, 64, 64, 3)

Model eğitimi başlıyor...
Epoch 1/2
7/7 ━━━━━━━━━━━━━━━━━━━━ 106s 2s/step - accuracy: 0.1239 - loss: 2.5090 - val_accuracy: 0.1500 - val_loss: 2.2957
Epoch 2/2
7/7 ━━━━━━━━━━━━━━━━━━━━ 16s 862ms/step - accuracy: 0.8789 - loss: 0.8322 - val_accuracy: 0.1500 - val_loss: 2.2730
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step

Doğrulama F1 Skoru (Macro): 0.037
Doğrulama F1 Skoru (Weighted): 0.039


=== MODEL KARŞILAŞTIRMA ANALİZİ ===

Model Parametre Karşılaştırması:
DenseNet-121 parametreleri: 8,062,504
ResNet-50 parametreleri: 25,636,712
DenseNet, ResNet-50'den %68.6 daha az parametre kullanıyor

Tüm örnekler tamamlandı! 🎉
